# Upstream drainage-area maps

Visual reasonability checks for a handful of example sensors: for each,
plot its actual upstream river network and drainage area (from
`RiverNetwork`), color it by upstream distance from the sensor, and
annotate it with the sensor's own `lc_*` land-cover shares from
`data/assembly/sensor_panel.parquet`. The check is whether the map "looks
like" the numbers -- a sensor the panel calls mining-dominated should
visibly sit downstream of a mining-affected catchment, a low-disturbance
sensor's catchment should look mostly intact, and so on.

Four examples are chosen programmatically to span the land-cover
categories the descriptive-statistics notebook (`pollution.ipynb`) groups
outcomes by: mining-, pasture-, and urban-dominated sensors, plus a
low-disturbance control.

This replaces the previous municipality-centered version of this notebook
(`upstream_mining_municipality_map.ipynb`), which selected an upstream
seed region from a GADM municipality boundary. Seeding directly from a
sensor's own `trench_id` is more directly useful for reasonability-checking
the assembled panel, and drops the dependency on the old
`data/land_cover/land_cover.feather` / `sensor_data.duckdb` / raw GADM
inputs, none of which exist under their old names/locations after the
`code/` → `src/` package rename. The upstream-distance resolver used here
(`src.data.shared.sensor_upstream.resolve_upstream_trench_distances`) is
the same one `src.data.sources.land_cover.assembly` uses to build
`land_cover_sensor_upstream.parquet`, so "what this notebook draws" and
"what the assembled panel's `lc_*`/`alr_*` columns were computed from" are
guaranteed to be the same computation.

**Expected inputs:**

- `data/river_network/processed/` -- a full `data preprocess --source
  river_network` output: `river_network_trenches.parquet`,
  `river_network_drainage_areas.parquet`, and critically
  `river_network_system_matrices.pkl` (the reachability/distance
  matrices the upstream BFS needs; ~8.7 GB). This is the same directory
  `RiverNetwork.save()`/`.load()` use by default
  (`processed_dir(".", "river_network")`), not `data/river_network/`
  itself. The load step below raises a clear error if any of these are
  missing.
- `data/assembly/sensor_panel.parquet`

Pulled from `schulz0022@transfer11.scicore.unibas.ch:.../data/river_network/processed/`
via rsync on 2026-09-14 (~10.4 GB) to populate the above locally.

In [ ]:
from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data.sources import river_network as rn_module
from src.data.sources.land_cover.schema import build_trench_length_lookup
from src.data.shared.sensor_upstream import (
    build_system_trench_lookup,
    build_trench_system_position_lookup,
    resolve_upstream_trench_distances,
    shift_upstream_distances,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)

In [ ]:
def locate_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


PROJECT_ROOT = locate_project_root()
# `RiverNetwork.save()`/`.load()` both go through `processed_dir(".",
# "river_network")` (src/data/sources/river_network/constants.py), i.e.
# `data/river_network/processed/`, not `data/river_network/` itself.
RIVER_NETWORK_DIR = PROJECT_ROOT / "data" / "river_network" / "processed"
SENSOR_PANEL_PATH = PROJECT_ROOT / "data" / "assembly" / "sensor_panel.parquet"
OUTPUT_FIGURE = PROJECT_ROOT / "output" / "figures" / "upstream_sensor_reasonability_maps.png"

# Land-cover columns pulled from the panel for both sensor selection and
# the on-map annotation. Update these before running if you'd rather split
# out `lc_farming_unclassified`/`lc_water` explicitly instead of leaving
# them implicit in "not otherwise accounted for".
LAND_COVER_COLS = [
    "lc_forest", "lc_nonforest_nat", "lc_pasture", "lc_agriculture",
    "lc_farming_unclassified", "lc_urban", "lc_mining", "lc_other", "lc_water",
]
FIGSIZE = (14, 14)
BUFFER_FACTOR = 0.12

PROJECT_ROOT

In [ ]:
def require_path(path: Path, label: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    return path


def buffered_bounds(geometries: gpd.GeoSeries, buffer_factor: float = 0.08) -> tuple[float, float, float, float]:
    xmin, ymin, xmax, ymax = geometries.total_bounds
    xpad = max((xmax - xmin) * buffer_factor, 0.05)
    ypad = max((ymax - ymin) * buffer_factor, 0.05)
    return xmin - xpad, xmax + xpad, ymin - ypad, ymax + ypad


def upstream_trenches_from_sensor(
    trench_id,
    network,
    trench_lengths,
    system_trench_id_arrays,
    system_valid_positions,
    trench_system_position_lookup,
):
    """Along-network upstream distance from one sensor's trench to every
    trench that drains into it, with distances shifted so 0 km is the
    sensor's own location (matching `land_cover_sensor_upstream.parquet`'s
    convention) rather than the far end of its trench segment.
    """
    upstream_distances = resolve_upstream_trench_distances(
        trench_id,
        network,
        system_trench_id_arrays,
        system_valid_positions,
        trench_system_position_lookup,
        trench_id_column=rn_module.TRENCH_ID_COLUMN,
        distance_column=rn_module.DISTANCE_COLUMN,
    )
    return shift_upstream_distances(
        upstream_distances,
        trench_lengths,
        trench_id_column=rn_module.TRENCH_ID_COLUMN,
        distance_column=rn_module.DISTANCE_COLUMN,
        adjusted_distance_column="adjusted_distance_km",
    )

In [ ]:
require_path(RIVER_NETWORK_DIR, "river network directory")
require_path(SENSOR_PANEL_PATH, "assembled sensor panel")

network = rn_module.RiverNetwork()
network.load(str(RIVER_NETWORK_DIR))
if network.trenches is None or network.drainage_areas is None or not network.trench_reachability_matrices:
    raise ValueError(
        "River network directory is missing one or more required outputs "
        f"(`{rn_module.TRENCHES_FILENAME}`, `{rn_module.DRAINAGE_AREAS_FILENAME}`, "
        f"`{rn_module.SYSTEM_MATRICES_FILENAME}`) under {RIVER_NETWORK_DIR}. "
        "This notebook needs a full `data preprocess --source river_network` output "
        "(trenches, drainage areas, and the system reachability/distance matrices), "
        "not just trench/drainage-area extracts."
    )

rivers = network.trenches
trench_lengths = build_trench_length_lookup(rivers)
system_trench_id_arrays, _system_trench_positions, system_valid_positions = build_system_trench_lookup(
    rivers, rn_module=rn_module, trench_id_column=rn_module.TRENCH_ID_COLUMN
)
trench_system_position_lookup = build_trench_system_position_lookup(
    rivers, rn_module=rn_module, trench_id_column=rn_module.TRENCH_ID_COLUMN
)

sensor_panel = pd.read_parquet(
    SENSOR_PANEL_PATH,
    columns=["station_code", "year", "trench_id", "biome", *LAND_COVER_COLS],
)
sensor_panel = sensor_panel.dropna(subset=["trench_id"])
# One row per station: its most recent land-cover snapshot.
sensor_latest = sensor_panel.sort_values("year").groupby("station_code").last()
sensor_latest["trench_id"] = sensor_latest["trench_id"].astype(np.int64)

sensor_latest.shape

## Example sensor selection

One example per land-cover-conversion story -- the same categories
`PASTURE_OUTCOMES`/`MINING_OUTCOMES` in `pollution.ipynb` group outcomes
by -- plus a low-disturbance control, picked as whichever station has the
highest current share of that category. This is fully reproducible against
whatever `sensor_panel.parquet` currently contains; the station codes
below were confirmed against the panel active at the time this notebook
was last adapted.

In [ ]:
EXAMPLE_SENSORS = {
    "mining-dominated": sensor_latest["lc_mining"].idxmax(),
    "pasture-dominated": sensor_latest["lc_pasture"].idxmax(),
    "urban-dominated": sensor_latest["lc_urban"].idxmax(),
    "low-disturbance (mostly natural land)": (
        sensor_latest["lc_forest"] + sensor_latest["lc_nonforest_nat"]
    ).idxmax(),
}
assert len(set(EXAMPLE_SENSORS.values())) == len(EXAMPLE_SENSORS), "expected 4 distinct example sensors"

sensor_latest.loc[list(EXAMPLE_SENSORS.values()), ["biome", "trench_id", *LAND_COVER_COLS]]

## Maps

Each panel: drainage-area polygons shaded by along-network distance
upstream of the sensor, upstream river segments in blue, the sensor's own
trench highlighted in red with a star marker, and a text box repeating the
panel's own land-cover shares for that station so the visual and the
number can be compared directly.

In [ ]:
def plot_sensor_upstream_map(ax, label, station_code, panel_row, network, trench_lengths, system_lookups):
    system_trench_id_arrays, system_valid_positions, trench_system_position_lookup = system_lookups
    trench_id = int(panel_row["trench_id"])

    upstream = upstream_trenches_from_sensor(
        trench_id,
        network,
        trench_lengths,
        system_trench_id_arrays,
        system_valid_positions,
        trench_system_position_lookup,
    )
    upstream_trench_ids = upstream[rn_module.TRENCH_ID_COLUMN].to_numpy()

    upstream_rivers = network.trenches.loc[
        network.trenches[rn_module.TRENCH_ID_COLUMN].astype(np.int64).isin(upstream_trench_ids)
    ].copy().to_crs(4326)
    upstream_drainage_areas = network.drainage_areas.loc[
        network.drainage_areas[rn_module.TRENCH_ID_COLUMN].astype(np.int64).isin(upstream_trench_ids)
    ].copy().to_crs(4326)
    upstream_drainage_areas = upstream_drainage_areas.merge(
        upstream, on=rn_module.TRENCH_ID_COLUMN, how="left", validate="one_to_one",
    )

    seed_geometry = network.trenches.loc[
        network.trenches[rn_module.TRENCH_ID_COLUMN].astype(np.int64) == trench_id
    ].to_crs(4326)

    plot_geometries = gpd.GeoDataFrame(
        pd.concat(
            [
                upstream_drainage_areas[["geometry"]],
                upstream_rivers[["geometry"]],
                seed_geometry[["geometry"]],
            ],
            ignore_index=True,
        ),
        geometry="geometry",
        crs=4326,
    )
    xmin, xmax, ymin, ymax = buffered_bounds(plot_geometries.geometry, BUFFER_FACTOR)

    upstream_drainage_areas.plot(
        ax=ax,
        column="adjusted_distance_km",
        cmap="YlGnBu",
        linewidth=0.1,
        edgecolor="#636363",
        legend=True,
        legend_kwds={"label": "Upstream distance (km)", "shrink": 0.7},
        missing_kwds={"color": "#f3f3f3", "edgecolor": "#bdbdbd"},
    )
    upstream_rivers.plot(ax=ax, color="#2c7fb8", linewidth=0.7, alpha=0.85, zorder=3)
    seed_geometry.plot(ax=ax, color="#d62728", linewidth=3.0, zorder=4)
    seed_point = seed_geometry.geometry.iloc[0].centroid
    ax.scatter([seed_point.x], [seed_point.y], color="#d62728", edgecolor="white", s=100, zorder=5, marker="*")

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(f"{label}\n{station_code} ({panel_row['biome']})", fontsize=11)

    summary_lines = [
        f"Upstream trenches: {len(upstream_trench_ids):,}",
        f"natural (forest+nonforest_nat): {panel_row['lc_forest'] + panel_row['lc_nonforest_nat']:.0%}",
        f"pasture: {panel_row['lc_pasture']:.0%}  agriculture: {panel_row['lc_agriculture']:.0%}",
        f"urban: {panel_row['lc_urban']:.0%}  mining: {panel_row['lc_mining']:.0%}",
    ]
    ax.text(
        0.02,
        0.02,
        "\n".join(summary_lines),
        transform=ax.transAxes,
        fontsize=8,
        ha="left",
        va="bottom",
        bbox={"facecolor": "white", "alpha": 0.9, "edgecolor": "none"},
    )
    return upstream

In [ ]:
system_lookups = (system_trench_id_arrays, system_valid_positions, trench_system_position_lookup)

fig, axes = plt.subplots(2, 2, figsize=FIGSIZE)
upstream_results = {}
for ax, (label, station_code) in zip(axes.flat, EXAMPLE_SENSORS.items()):
    panel_row = sensor_latest.loc[station_code]
    upstream_results[station_code] = plot_sensor_upstream_map(
        ax, label, station_code, panel_row, network, trench_lengths, system_lookups
    )

fig.suptitle(
    "Upstream drainage-area reasonability checks: does the map match the panel's land-cover shares?",
    fontsize=13,
)
plt.tight_layout()
plt.show()

In [ ]:
OUTPUT_FIGURE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_FIGURE, dpi=300, bbox_inches="tight")
OUTPUT_FIGURE